In [ ]:
# General libraries
import os
import re
import random
import pickle
import statistics
import collections
from collections import Counter
from itertools import combinations

# Data handling
import numpy as np
import pandas as pd

# Visualization
import seaborn as sns
from matplotlib import pyplot as plt, cm, colors, colorbar
from matplotlib_venn import venn2
from mpl_toolkits.axes_grid1 import make_axes_locatable
from adjustText import adjust_text

# Machine learning & preprocessing
from sklearn.linear_model import LinearRegression, RidgeClassifier, LogisticRegression, SGDClassifier
from sklearn.neural_network import MLPClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import AdaBoostClassifier, GradientBoostingClassifier, RandomForestClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.svm import SVC
from sklearn.cluster import KMeans, DBSCAN
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler, MinMaxScaler
from sklearn.feature_selection import SelectFromModel
from sklearn.manifold import TSNE
from sklearn.model_selection import LeaveOneOut, StratifiedKFold, train_test_split, cross_val_score, GridSearchCV
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    roc_auc_score, confusion_matrix, silhouette_score,
    davies_bouldin_score, calinski_harabasz_score, pairwise_distances
)

# Dimensionality reduction
import umap
import umap.umap_ as umap_module  # if you need the lower-level API

# Feature selection
from boruta import BorutaPy

# Shapelet learning
from pyts.classification import LearningShapelets
from pyts.datasets import load_gunpoint
from pyts.utils import windowed_view

# Statistical tests and models
import statsmodels.api as sm
from statsmodels.stats.multitest import multipletests
from statannot import add_stat_annotation
#import scikit_posthocs as sp
import shap
#from xgboost import XGBRegressor
from scipy import stats
from scipy.stats import (
    ttest_ind, levene, mannwhitneyu, shapiro, mstats,
    pearsonr, kruskal, skew
)
from scipy.spatial import ConvexHull, convex_hull_plot_2d

# Progress bar
from tqdm import tqdm
from statsmodels.formula.api import ols
import warnings
from tqdm import tqdm
warnings.filterwarnings('ignore') 
from scipy.stats import chi2_contingency, mannwhitneyu, kruskal, ttest_ind, f_oneway
from statsmodels.stats.multitest import multipletests
import warnings
from scipy.stats import chi2_contingency
from collections import Counter
# ----------------------------------------------------------------------
# Example initializations (optional)
scaler = StandardScaler()
mmscaler = MinMaxScaler()
pca = PCA(n_components=2, svd_solver='full')

# Classifiers
rdg = RidgeClassifier(alpha=0.5)
mlp = MLPClassifier(random_state=1, max_iter=300, activation='relu')
lgr = LogisticRegression(random_state=1, max_iter=500)
DT = DecisionTreeClassifier(random_state=0, max_depth=10)
adb = AdaBoostClassifier(n_estimators=100, random_state=0)
gbc = GradientBoostingClassifier(n_estimators=100, random_state=1)
knn = KNeighborsClassifier(n_neighbors=3)
SGD = SGDClassifier(loss='log', random_state=1, max_iter=100, early_stopping=True,
                    learning_rate='optimal', validation_fraction=0.2)
rf = RandomForestClassifier(max_depth=10, random_state=0)
clf_svm = SVC(kernel='rbf')

In [ ]:
def extract_connectivity(band,data):
    Y=[]
    coh_ar=np.zeros([len(data),88*88])
    for i in range(0,len(data)):
        m=np.loadtxt(data[i])[band*88:(band+1)*88,:] # extract only delta band 
        m=np.tril(m, k=-1).flatten()  ## Take upper/lower Triangle of the Symetrical Coherence Matrix
        coh_ar[i,:]=m
    coh_ar_zscored = stats.zscore(coh_ar, axis=0) # Within Subject Z-Transform
    return coh_ar_zscored

org_directory="/home/jupy/Data_SourceFC/T0"
ls_org_directory=os.listdir(org_directory)
directory_T0=[item for item in ls_org_directory if item !='.ipynb_checkpoints' ]
directory_T0_ob=[x for x in directory_T0 if x.startswith("F")]
directory_T0_lean=[x for x in directory_T0 if x.startswith("L")]

org_directory="/home/jupy/Data_SourceFC/T45"
ls_org_directory=os.listdir(org_directory)
directory_T45=[item for item in ls_org_directory if item != '.ipynb_checkpoints']
directory_T45_ob=[x for x in directory_T45 if x.startswith("F")]
directory_T45_lean=[x for x in directory_T45 if x.startswith("L")]

In [ ]:
def extract_key(filename):
    return filename[0:4]
    
def align_subjects_btw_T0_T45 (directory_list_T0, directory_list_T45):
    # Create dictionaries for all time points
    time_points = {
        'T0': {extract_key(f): f for f in directory_list_T0 if extract_key(f)},
        'T45': {extract_key(f): f for f in directory_list_T45 if extract_key(f)}}    
    # Find common keys across ALL time points
    common_keys = set(time_points['T0'])  # Start with T0 keys
    for tp in time_points:
        common_keys &= set(time_points[tp])  # Intersect with each time point
        # Extract aligned files for each time point (sorted by key)
    aligned_files = {
        tp: [time_points[tp][key] for key in sorted(common_keys)]
        for tp in time_points }
    # Find unaligned files for each time point
    unique_files = {
        tp: [f for key, f in time_points[tp].items() if key not in common_keys]
        for tp in time_points}
    
    excl0=unique_files['T0']
    aligned_directory_T0=[item for item in directory_list_T0 if item not in excl0 ]
    excl45=unique_files['T45']
    aligned_directory_T45=[item for item in directory_list_T45 if item not in excl45]
    print ('Length T0: ',len(aligned_directory_T0),'    Length T45',len(aligned_directory_T45)) 
    
    # Create dictionaries with 4-digit keys
    dict_T0 = {extract_key(f): f for f in aligned_directory_T0 if extract_key(f)}
    dict_T45 = {extract_key(f): f for f in aligned_directory_T45 if extract_key(f)}
    # Find aligned keys
    common_keys = set(dict_T0) & set(dict_T45)
    aligned_T0 = [dict_T0[key] for key in common_keys]
    aligned_T45 = [dict_T45[key] for key in common_keys]
    # Find unaligned elements
    unique_T0 = [f for key, f in dict_T0.items() if key not in common_keys]
    unique_T45 = [f for key, f in dict_T45.items() if key not in common_keys]
    # Sort both aligned lists by key (first 4 digits)
    aligned_pairs = sorted(zip(aligned_T0, aligned_T45), key=lambda x: extract_key(x[0]))
    aligned_T0_sorted, aligned_T45_sorted = zip(*aligned_pairs) if aligned_pairs else ([], [])
    # Final sorted lists (now aligned by first 4 digits)
    finaldir_T0 = list(aligned_T0_sorted) 
    finaldir_T45 = list(aligned_T45_sorted) 
    
    return finaldir_T0, finaldir_T45


In [ ]:
finaldir_T0_lean,finaldir_T45_lean=align_subjects_btw_T0_T45 (directory_T0_lean, directory_T45_lean)
finaldir_T0_ob,finaldir_T45_ob=align_subjects_btw_T0_T45 (directory_T0_ob, directory_T45_ob)

In [ ]:
################  Use If Analyse Each Individual Band
def get_fc_data_per_T(fc_data_dir, subject_list_dir, band="alpha"):
    os.chdir(fc_data_dir)
    band_map = {
        "delta": 0,
        "theta": 1,
        "alpha": 2,
        "beta": 3,
        "gamma": 4 }  
    
    if band not in band_map:
        raise ValueError(f"Invalid band '{band}'. Choose from {list(band_map.keys())}.")  
    fc_data = extract_connectivity(band_map[band], subject_list_dir)
    return fc_data

#### Within Band Per Subject Z-transform
Study to back up **within band standardization** (Z-transform)
<br>EEG Frequency Bands in Psychiatric Disorders: A Review of Resting State Studies (https://www.frontiersin.org/journals/human-neuroscience/articles/10.3389/fnhum.2018.00521/full)
<br>*"We emphasize the need to use a standardized definition for each frequency band, based on the most commonly used non-overlapping frequencies: (delta: <4 Hz; theta: 4–7.5 Hz; alpha: 7.5–12.5 Hz; beta: 12.5–30 Hz; gamma: 30–40 Hz)."


In [ ]:
band='gamma'
bands=[band]

In [ ]:
fc_T0_ob=get_fc_data_per_T ("/home/jupy/Data_SourceFC/T0",finaldir_T0_ob,band)
fc_T45_ob=get_fc_data_per_T ("/home/jupy/Data_SourceFC/T45",finaldir_T45_ob,band)
fc_T0_lean=get_fc_data_per_T ("/home/jupy/Data_SourceFC/T0",finaldir_T0_lean,band)
fc_T45_lean=get_fc_data_per_T ("/home/jupy/Data_SourceFC/T45",finaldir_T45_lean,band)

### ----------------- Extract FC with ROI Names ------------------

In [ ]:
# Define ROIs and frequency bands
rois = [f'roi{i+1}' for i in range(88)]  
print(bands)

column_names = []
for band in bands:
    for source_roi in rois:
        for target_roi in rois:
            column_names.append(f"{source_roi}_{target_roi}_{band}")

# Verify we have the right number of features: 5 bands * 88 source ROIs * 88 target ROIs 
print(f"Generated {len(column_names)} column names")

fc_T45_ob= pd.DataFrame(fc_T45_ob, columns=column_names).abs()
fc_T0_ob= pd.DataFrame(fc_T0_ob, columns=column_names).abs()
fc_T45_lean= pd.DataFrame(fc_T45_lean, columns=column_names).abs()
fc_T0_lean= pd.DataFrame(fc_T0_lean, columns=column_names).abs()

def extract_roi_columns_regex(fc_df, roi_numbers): #Extract columns where column names contain any of the specified ROI numbers
    # Remove 0 from the list since your columns start from roi1
    roi_numbers = [roi for roi in roi_numbers if roi != 0]    
    # Create regex pattern to match roiX where X is any of the numbers
    pattern = r'roi(' + '|'.join(map(str, roi_numbers)) + r')(_|$)'
    # Filter columns that match the pattern
    matching_columns = [
        col for col in fc_df.columns 
        if re.search(pattern, col) ]
    return fc_df[matching_columns]

#acc_roi_numbers = [1, 20, 21, 42, 43, 54, 55, 56, 57, 86, 88]
#acc_roi_numbers=[85, 41, 53, 42, 87, 19, 20, 54, 55, 56]
acc_roi_numbers=np.arange(1,89).tolist()

acc_fc_T45_ob = extract_roi_columns_regex(fc_T45_ob, acc_roi_numbers)
acc_fc_T0_ob = extract_roi_columns_regex(fc_T0_ob, acc_roi_numbers)
acc_fc_T45_lean = extract_roi_columns_regex(fc_T45_lean, acc_roi_numbers)
acc_fc_T0_lean = extract_roi_columns_regex(fc_T0_lean, acc_roi_numbers)

compute **weighted degree connectivity** (i.e. strength) 

In [ ]:
def calculate_specific_roi_connectivity_sum_efficient(acc_fcdf, roi_list):
    # Convert ROI numbers to the format used in column names
    roi_strings = [f"roi{roi}" for roi in roi_list]
    # Create empty result dataframe
    result_df = pd.DataFrame(0, index=acc_fcdf.index, columns=roi_strings)
    # Pre-process column information
    column_info = []
    for col in acc_fcdf.columns:
        parts = col.split('_')
        roi1, roi2 = parts[0], parts[1]
        column_info.append((roi1, roi2, col))
    # For each target ROI, sum all connections involving it
    for roi in roi_strings:
        # Find all columns where this ROI appears
        roi_columns = []
        for roi1, roi2, col_name in column_info:
            if roi1 == roi or roi2 == roi: ########### Self-connection excluded , Symmetrical pair only cont one of the unique #################
                roi_columns.append(col_name) 
        # Sum all connections involving this ROI
        if roi_columns:
            result_df[roi] = acc_fcdf[roi_columns].sum(axis=1)  
    return result_df

In [ ]:
roi_connectivity_T0_ob = calculate_specific_roi_connectivity_sum_efficient(acc_fc_T0_ob, acc_roi_numbers)
roi_connectivity_T45_ob = calculate_specific_roi_connectivity_sum_efficient(acc_fc_T45_ob, acc_roi_numbers)
roi_connectivity_T0_lean = calculate_specific_roi_connectivity_sum_efficient(acc_fc_T0_lean, acc_roi_numbers)
roi_connectivity_T45_lean = calculate_specific_roi_connectivity_sum_efficient(acc_fc_T45_lean, acc_roi_numbers)

In [ ]:
from scipy.stats import zscore

roi_connectivity_T0_ob = (roi_connectivity_T0_ob).apply(zscore)
roi_connectivity_T45_ob = (roi_connectivity_T45_ob ).apply(zscore)
roi_connectivity_T0_lean = (roi_connectivity_T0_lean ).apply(zscore)
roi_connectivity_T45_lean =(roi_connectivity_T45_lean).apply(zscore)

### ANCOVA 

`model.resid`: These are the residuals – the part of the T45 value that the model could not explain using T0. They represent the "adjusted T45" values after removing the linear influence of T0.
<br>`model.params['Intercept']`: Addthe intercept back to "re-center" the residuals around the grand mean of T45. Without this, the residuals would have a mean of zero. Adding the intercept back creates values that are on the original scale, which is much more intuitive for plotting and further analysis (e.g., group comparisons).

In [ ]:
from statsmodels.formula.api import ols
from statsmodels.stats.diagnostic import het_breuschpagan

connections = roi_connectivity_T0_lean.columns
groups = {'Lean': (roi_connectivity_T0_lean, roi_connectivity_T45_lean),
          'Obese': (roi_connectivity_T0_ob, roi_connectivity_T45_ob)}

# DataFrames to store assumption check results
assumption_results = pd.DataFrame(index=connections, 
                                  columns=pd.MultiIndex.from_product([['Lean', 'Obese'], 
                                                                     ['Linear_pval', 'Normality_pval', 'Homoscedasticity_pval', 'Slope']]))

# DataFrames to store the FINAL adjusted T45 values
t45_adjusted_lean = pd.DataFrame(index=roi_connectivity_T45_lean.index)
t45_adjusted_obese = pd.DataFrame(index=roi_connectivity_T45_ob.index)

In [ ]:
# Function to check assumptions and adjust data
def check_assumptions_and_adjust(t0_data, t45_data, group_name, conn):
    """
    Checks assumptions for a single connection within a single group.
    Returns adjusted T45 values and assumption results.
    """
    df = pd.DataFrame({'T0': t0_data, 'T45': t45_data}).dropna()
    if len(df) < 3:  # Need at least 3 points for regression
        return np.nan, {'Linear_pval': np.nan, 'Normality_pval': np.nan, 
                       'Homoscedasticity_pval': np.nan, 'Slope': np.nan}
    
    # Fit the model
    model = ols('T45 ~ T0', data=df).fit()
    
    # Get predictions and residuals
    predicted = model.fittedvalues
    residuals = model.resid
    
    # 1. Check Linearity (Using Rainbow test)
    try:
        # The rainbow test checks if the linear fit is adequate
        rainbow_stat, rainbow_pval = sm.stats.diagnostic.linear_rainbow(model)
    except:
        rainbow_pval = np.nan
    
    # 2. Check Normality of Residuals (Shapiro-Wilk test)
    if len(residuals) > 3:  # Shapiro-Wilk requires 3+ samples
        _, normality_pval = stats.shapiro(residuals)
    else:
        normality_pval = np.nan
    
    # 3. Check Homoscedasticity (Breusch-Pagan test)
    try:
        # The Breusch-Pagan test checks if variance of residuals is constant
        bp_lm, bp_pval, _, _ = het_breuschpagan(residuals, model.model.exog)
    except:
        bp_pval = np.nan
    
    # Calculate adjusted T45 values
    adjusted_values = residuals + model.params['Intercept']
    
    results = {
        'Linear_pval': rainbow_pval,
        'Normality_pval': normality_pval,
        'Homoscedasticity_pval': bp_pval,
        'Slope': model.params['T0']
    }
    
    return adjusted_values, results

In [ ]:
for idx, (group_name, (t0_df, t45_df)) in enumerate(groups.items()):
    print(f"\nProcessing {group_name} group...")
    
    for conn in connections:
        # Get data for this connection and group
        t0_data = t0_df[conn]
        t45_data = t45_df[conn]
        
        # Check assumptions and get adjusted values
        adjusted_values, results = check_assumptions_and_adjust(t0_data, t45_data, group_name, conn)
        
        # Store results
        for key, value in results.items():
            assumption_results.loc[conn, (group_name, key)] = value
        
        # Store adjusted values
        if group_name == 'Lean':
            t45_adjusted_lean[conn] = adjusted_values
        else:
            t45_adjusted_obese[conn] = adjusted_values
        
# --------------------------------------------------------------------
# SUMMARIZE ASSUMPTION VIOLATIONS
# --------------------------------------------------------------------
print("\n" + "="*60)
print("ASSUMPTION CHECK SUMMARY")
print("="*60)

alpha = 0.05
for group in ['Lean', 'Obese']:
    print(f"\n--- {group} Group ---")
    linear_violations = (assumption_results[(group, 'Linear_pval')] < alpha).sum()
    normality_violations = (assumption_results[(group, 'Normality_pval')] < alpha).sum()
    homosced_violations = (assumption_results[(group, 'Homoscedasticity_pval')] < alpha).sum()
    
    print(f"Connections violating Linearity (Rainbow test p < {alpha}): {linear_violations}/{len(connections)}")
    print(f"Connections violating Normality (Shapiro-Wilk p < {alpha}): {normality_violations}/{len(connections)}")
    print(f"Connections violating Homoscedasticity (Breusch-Pagan p < {alpha}): {homosced_violations}/{len(connections)}")

In [ ]:
# --------------------------------------------------------------------
# ADDRESS VIOLATIONS - ROBUST ADJUSTMENT
# --------------------------------------------------------------------
print("\n" + "="*60)
print("ADDRESSING ASSUMPTION VIOLATIONS")
print("="*60)

# Create new DataFrames for robustly adjusted values
t45_adjusted_robust_lean = pd.DataFrame(index=roi_connectivity_T45_lean.index)
t45_adjusted_robust_obese = pd.DataFrame(index=roi_connectivity_T45_ob.index)

# Apply robust methods for connections with violations
for conn in connections:
    for group_name, (t0_df, t45_df) in groups.items():
        t0_data = t0_df[conn]
        t45_data = t45_df[conn]
        df = pd.DataFrame({'T0': t0_data, 'T45': t45_data}).dropna()
        
        if len(df) < 3:
            continue
            
        # Check if this connection has severe violations
        linear_pval = assumption_results.loc[conn, (group_name, 'Linear_pval')]
        norm_pval = assumption_results.loc[conn, (group_name, 'Normality_pval')]
        hets_pval = assumption_results.loc[conn, (group_name, 'Homoscedasticity_pval')]
        
        # If severe linearity violation, use non-parametric adjustment
        if pd.notna(linear_pval) and linear_pval < 0.01:
            # Use rank-based transformation
            df['T0_rank'] = df['T0'].rank()
            df['T45_rank'] = df['T45'].rank()
            model = ols('T45_rank ~ T0_rank', data=df).fit()
            adjusted_ranks = model.resid + model.params['Intercept']
            # Convert back to original scale using quantiles
            adjusted_values = adjusted_ranks.rank(pct=True).apply(lambda x: np.quantile(df['T45'], x))
            
        # If severe non-normality or heteroscedasticity, use robust regression
        elif (pd.notna(norm_pval) and norm_pval < 0.01) or (pd.notna(hets_pval) and hets_pval < 0.01):
            # Use Huber robust regression
            huber_t = sm.RLM(df['T45'], sm.add_constant(df['T0']), M=sm.robust.norms.HuberT())
            huber_results = huber_t.fit()
            predicted = huber_results.predict(sm.add_constant(df['T0']))
            residuals = df['T45'] - predicted
            adjusted_values = residuals + huber_results.params[0]  # Add intercept
            
        else:
            # Use standard OLS adjustment
            model = ols('T45 ~ T0', data=df).fit()
            adjusted_values = model.resid + model.params['Intercept']
        
        # Store robustly adjusted values
        if group_name == 'Lean':
            t45_adjusted_robust_lean[conn] = adjusted_values
        else:
            t45_adjusted_robust_obese[conn] = adjusted_values

# --------------------------------------------------------------------
# SAVE THE RESULTS
# --------------------------------------------------------------------
print (f'### Current Band: {band} ###')
# Save assumption results
#assumption_results.to_csv(f'/home/jupy/phenotype_fc_hormones/Data_Ancova_Adj_WDFC/{band}_Within_Group_Assumption_Check_Results.csv')
#t45_adjusted_robust_lean.to_csv(f'/home/jupy/phenotype_fc_hormones/Data_Ancova_Adj_WDFC/{band}_T45_Adjusted_Robust_Lean.csv')
#t45_adjusted_robust_obese.to_csv(f'/home/jupy/phenotype_fc_hormones/Data_Ancova_Adj_WDFC/{band}_T45_Adjusted_Robust_Obese.csv')

print(f"\nFinal adjusted datasets ready for within-group analysis.")
print(f"Lean group shape: {t45_adjusted_robust_lean.shape}")
print(f"Obese group shape: {t45_adjusted_robust_obese.shape}")

adjusted_fc_ob=t45_adjusted_robust_obese 
adjusted_fc_lean=t45_adjusted_robust_lean

## Robustness Test of Umap and Clustering

In [ ]:
from sklearn.metrics import adjusted_rand_score
from scipy.spatial import procrustes
from scipy.stats import mode
from umap.umap_ import UMAP as UMAP

df=adjusted_fc_ob
X = df.values

In [ ]:
print("\n=== Step 1: UMAP Hyperparameter Exploration ===")
n_neighbors_list = np.arange(3, 21, 2)
min_dist_list = np.arange(0.05,1,0.05)

param_results = []
for n_neighbors in n_neighbors_list:
    for min_dist in min_dist_list:
        # Run UMAP with fixed random state for consistent comparison
        reducer = UMAP(n_neighbors=n_neighbors, 
                           min_dist=min_dist, 
                           random_state=42,
                           n_components=2)
        embedding = reducer.fit_transform(X)
        
        # Calculate basic statistics about the embedding
        param_results.append({
            'n_neighbors': n_neighbors,
            'min_dist': min_dist,
            'embedding_mean_x': np.mean(embedding[:, 0]),
            'embedding_std_x': np.std(embedding[:, 0]),
            'embedding_mean_y': np.mean(embedding[:, 1]),
            'embedding_std_y': np.std(embedding[:, 1])
        })


# Save parameter exploration results
param_df = pd.DataFrame(param_results)

In [ ]:
os.chdir('/home/jupy/Subtypes_Obesity_Clustering')

os.makedirs(f'{band}_RobustTest_clustering_results', exist_ok=True)
param_df.to_csv(f'{band}_RobustTest_clustering_results/umap_parameter_exploration.txt', sep='\t', index=False)

In [ ]:
# Visualisation to chose the best n_neighbours and min_dist

def visualize_umap_parameters(X, n_neighbors_list, min_dist_list, param_df):
    """Visualize UMAP embeddings for different parameter combinations."""
    
    # Dynamically scale figure size based on number of subplots
    fig_width = 4 * len(min_dist_list)
    fig_height = 4 * len(n_neighbors_list)
    fig, axes = plt.subplots(len(n_neighbors_list), len(min_dist_list), 
                             figsize=(fig_width, fig_height),
                             squeeze=False)
    
    fig.suptitle('UMAP Parameter Exploration - Visual Assessment', 
                 fontsize=18, fontweight='bold')
    
    plot_data = []
    
    for i, n_neighbors in enumerate(n_neighbors_list):
        for j, min_dist in enumerate(min_dist_list):
            # Find corresponding parameter combination
            param_row = param_df[
                (param_df['n_neighbors'] == n_neighbors) & 
                (param_df['min_dist'] == min_dist)
            ]
            if len(param_row) == 0:
                axes[i, j].axis('off')
                continue
            
            # Generate embedding
            reducer = UMAP(
                n_neighbors=n_neighbors, 
                min_dist=min_dist, 
                random_state=42,
                n_components=2
            )
            embedding = reducer.fit_transform(X)
            
            # Plot scatter — adjust size and transparency for readability
            ax = axes[i, j]
            ax.scatter(embedding[:, 0], embedding[:, 1], 
                       s=13, alpha=0.6, linewidth=0,color='black')
            
            # Adjust title font size based on number of panels
            title_font = 12 if len(n_neighbors_list) * len(min_dist_list) > 4 else 14
            ax.set_title(f'n_neighbors={n_neighbors}\nmin_dist={min_dist}',
                         fontsize=title_font)
            
            # Clean up axis appearance
            ax.set_xticks([])
            ax.set_yticks([])
            ax.set_xlabel('')
            ax.set_ylabel('')
            
            # Store data for return
            plot_data.append({
                'n_neighbors': n_neighbors,
                'min_dist': min_dist,
                'embedding': embedding,
                'axis': ax
            })
    
    plt.tight_layout(rect=[0, 0, 1, 0.97])
    plt.savefig(f'{band}_RobustTest_clustering_results/{band}_umap_parameter_visualization.png', 
                dpi=300, bbox_inches='tight')
    plt.close(fig)
    
    return plot_data


In [ ]:
n_neighbors_list = sorted(param_df ['n_neighbors'].unique())
min_dist_list = sorted(param_df ['min_dist'].unique())
#plot_data = visualize_umap_parameters(X, n_neighbors_list, min_dist_list, param_df)

# delta
optimal_n_neighbors = 17
optimal_min_dist = 0.15
# theta
optimal_n_neighbors = 11
optimal_min_dist = 0.9
# alpha
optimal_n_neighbors = 11
optimal_min_dist = 0.15
# beta
optimal_n_neighbors = 11
optimal_min_dist = 0.35
# gamma
optimal_n_neighbors = 15
optimal_min_dist = 0.1

In [ ]:
band

In [ ]:
optimal_n_neighbors = 15
optimal_min_dist = 0.1

In [ ]:
# Step 2: Choose optimal parameters and create primary embedding
print("\n=== Step 2: Primary UMAP Embedding ===")
reducer_primary =UMAP(n_neighbors=optimal_n_neighbors,
                           min_dist=optimal_min_dist,
                           random_state=42,
                           n_components=2)
primary_embedding = reducer_primary.fit_transform(X)
np.savetxt(f'{band}_RobustTest_clustering_results/primary_umap_embedding.txt', primary_embedding)

In [ ]:
# Step 3: Determine optimal k for clustering
print("\n=== Step 3: Determine Optimal k ===")
k_range = range(1, 10)
wcss = []
silhouette_scores = []

for k in k_range:
    kmeans = KMeans(n_clusters=k, random_state=42, n_init=10)
    cluster_labels = kmeans.fit_predict(primary_embedding)
    wcss.append(kmeans.inertia_)
    
    if k > 1:  # Silhouette score requires at least 2 clusters
        sil_score = silhouette_score(primary_embedding, cluster_labels)
        silhouette_scores.append(sil_score)
        print(f"k={k}: WCSS={kmeans.inertia_:.2f}, Silhouette={sil_score:.3f}")
    else:
        silhouette_scores.append(np.nan)
        print(f"k={k}: WCSS={kmeans.inertia_:.2f}")

# Save results
k_results = pd.DataFrame({
    'k': list(k_range),
    'wcss': wcss,
    'silhouette': silhouette_scores
})

k_results.to_csv(f'{band}_RobustTest_clustering_results/k_determination_results.txt', 
                 sep='\t', index=False)
print("\nSaved k determination results.")


In [ ]:
fig = plt.figure(figsize=(5, 3))

# Plot 1: Elbow Method
plt.subplot(1, 2, 1)  
plt.plot(k_range, wcss, 'bo-')
plt.xlabel('Number of clusters (k)')
plt.ylabel('Within-Cluster Sum of Squares')
plt.title('Elbow Method')

# Plot 2: Silhouette scores
plt.subplot(1, 2, 2)  # 2 rows, 1 column, second subplot
plt.plot(k_range, silhouette_scores, 'ro-')
plt.xlabel('Number of clusters (k)')
plt.ylabel('Silhouette Score')
plt.title('Silhouette Analysis')

plt.tight_layout()
plt.show()

In [ ]:
optimal_k = 2 # delta:2, theta: 3 , alpha: 2, beta:2 , gamma:2
print(f"Chosen k: {optimal_k}")

# Step 4: Primary Clustering
print("\n=== Step 4: Primary Clustering ===")
kmeans_primary = KMeans(n_clusters=optimal_k, random_state=42)
primary_cluster_labels = kmeans_primary.fit_predict(primary_embedding)

# Save primary clustering results
np.savetxt(f'{band}_RobustTest_clustering_results/primary_cluster_labels.txt', 
           primary_cluster_labels, fmt='%d')

In [ ]:
# Step 5: UMAP Stability Analysis
print("\n=== Step 5: UMAP Stability Analysis ===")
n_umap_iterations = 50
umap_embeddings = []
procrustes_errors = []

# Generate multiple UMAP embeddings with different random states
for i in range(n_umap_iterations):
    reducer = UMAP(n_neighbors=optimal_n_neighbors,
                       min_dist=optimal_min_dist,
                       random_state=i,  # Different seed for each iteration
                       n_components=2)
    embedding = reducer.fit_transform(X)
    umap_embeddings.append(embedding)
    
    # Calculate Procrustes error compared to primary embedding
    if i > 0:
        # Align current embedding to primary embedding using Procrustes
        mtx1, mtx2, disparity = procrustes(primary_embedding, embedding)
        procrustes_errors.append(disparity)

print(f"Generated {n_umap_iterations} UMAP embeddings")
print(f"Mean Procrustes error: {np.mean(procrustes_errors):.4f}")

# Save UMAP stability results
stability_results = {
    'n_iterations': n_umap_iterations,
    'mean_procrustes_error': np.mean(procrustes_errors),
    'std_procrustes_error': np.std(procrustes_errors)
}
pd.DataFrame([stability_results]).to_csv(f'{band}_RobustTest_clustering_results/umap_stability_results.txt', 
                                        sep='\t', index=False)
np.savetxt(f'{band}_RobustTest_clustering_results/procrustes_errors.txt', procrustes_errors)


In [ ]:
# Step 6: Full Pipeline Stability (UMAP + Clustering)
print("\n=== Step 6: Full Pipeline Stability Analysis ===")
all_cluster_labels = []
ari_scores = []

# Cluster each UMAP embedding
for i, embedding in enumerate(umap_embeddings):
    kmeans = KMeans(n_clusters=optimal_k, random_state=42, n_init=10)
    cluster_labels = kmeans.fit_predict(embedding)
    all_cluster_labels.append(cluster_labels)
    
    # Calculate ARI with primary clustering
    if i > 0:
        ari = adjusted_rand_score(primary_cluster_labels, cluster_labels)
        ari_scores.append(ari)

print(f"Mean ARI across all iterations: {np.mean(ari_scores):.4f} (+/- {np.std(ari_scores):.4f})")

# Save pipeline stability results
pipeline_stability = {
    'mean_ari': np.mean(ari_scores),
    'std_ari': np.std(ari_scores),
    'min_ari': np.min(ari_scores),
    'max_ari': np.max(ari_scores)
}
pd.DataFrame([pipeline_stability]).to_csv(f'{band}_RobustTest_clustering_results/pipeline_stability_results.txt', 
                                         sep='\t', index=False)
np.savetxt(f'{band}_RobustTest_clustering_results/ari_scores.txt', ari_scores)
print("Saved pipeline stability results")

In [ ]:
# Step 7: Consensus Clustering 
print("\n=== Step 7: Consensus Clustering ===")
# Create consensus labels by taking the mode across all iterations
all_labels_array = np.array(all_cluster_labels)
consensus_labels = mode(all_labels_array, axis=0)[0].flatten()

# Save consensus labels
np.savetxt(f'{band}_RobustTest_clustering_results/consensus_cluster_labels.txt', 
           consensus_labels, fmt='%d')

# Calculate ARI between consensus and primary labels
consensus_ari = adjusted_rand_score(primary_cluster_labels, consensus_labels)
print(f"ARI between primary and consensus labels: {consensus_ari:.4f}")

# Save consensus comparison
consensus_results = {
    'consensus_ari': consensus_ari
}
pd.DataFrame([consensus_results]).to_csv(f'{band}_RobustTest_clustering_results/consensus_results.txt', 
                                        sep='\t', index=False)
print("Saved consensus clustering results")

In [ ]:
# Step 8: Visualization
print("\n=== Step 8: Creating Visualizations ===")
plt.figure(figsize=(15, 10))

# Plot 1: Primary embedding with clusters
plt.subplot(2, 3, 1)
scatter = plt.scatter(primary_embedding[:, 0], primary_embedding[:, 1], 
                     c=primary_cluster_labels, cmap='viridis', s=50)
plt.colorbar(scatter)
plt.title('Primary UMAP + K-means Clustering')
plt.xlabel('UMAP 1')
plt.ylabel('UMAP 2')

# Plot 2: Elbow method
plt.subplot(2, 3, 2)
plt.plot(k_range, wcss, 'bo-')
plt.xlabel('Number of clusters (k)')
plt.ylabel('Within-Cluster Sum of Squares')
plt.title('Elbow Method')

# Plot 3: Silhouette scores
plt.subplot(2, 3, 3)
plt.plot(k_range, silhouette_scores, 'ro-')
plt.xlabel('Number of clusters (k)')
plt.ylabel('Silhouette Score')
plt.title('Silhouette Analysis')

# Plot 4: UMAP stability (Procrustes errors)
plt.subplot(2, 3, 4)
plt.hist(procrustes_errors, bins=15, alpha=0.7)
plt.xlabel('Procrustes Error')
plt.ylabel('Frequency')
plt.title('UMAP Stability')

# Plot 5: Pipeline stability (ARI scores)
plt.subplot(2, 3, 5)
plt.hist(ari_scores, bins=15, alpha=0.7)
plt.xlabel('Adjusted Rand Index')
plt.ylabel('Frequency')
plt.title('Pipeline Stability')

# Plot 6: Consensus vs Primary clustering
plt.subplot(2, 3, 6)
plt.scatter(primary_embedding[:, 0], primary_embedding[:, 1], 
           c=consensus_labels, cmap='viridis', s=50)
plt.colorbar()
plt.title('Consensus Clustering')
plt.xlabel('UMAP 1')
plt.ylabel('UMAP 2')

plt.tight_layout()
plt.savefig(f'{band}_RobustTest_clustering_results/clustering_analysis_summary.png', dpi=300, bbox_inches='tight')
plt.close()

print("Saved summary visualization")

In [ ]:
# Step 9: Test Clustering Method Stability
print("\n=== Step 9: Clustering Method Stability ===")

from sklearn.cluster import AgglomerativeClustering, DBSCAN
from sklearn.mixture import GaussianMixture
from sklearn.metrics import adjusted_rand_score

method_stability_results = []

# Method 1: Hierarchical Clustering (Ward)
print("Testing Hierarchical Clustering...")
ward = AgglomerativeClustering(n_clusters=optimal_k, linkage='ward')
ward_labels = ward.fit_predict(primary_embedding)
ward_ari = adjusted_rand_score(primary_cluster_labels, ward_labels)
method_stability_results.append({'method': 'Ward_Hierarchical', 'ARI': ward_ari})
print(f"Ward Hierarchical ARI: {ward_ari:.4f}")

# Method 2: Hierarchical Clustering (Average)
avg_linkage = AgglomerativeClustering(n_clusters=optimal_k, linkage='average')
avg_labels = avg_linkage.fit_predict(primary_embedding)
avg_ari = adjusted_rand_score(primary_cluster_labels, avg_labels)
method_stability_results.append({'method': 'Average_Hierarchical', 'ARI': avg_ari})
print(f"Average Linkage ARI: {avg_ari:.4f}")

# Method 3: Gaussian Mixture Models
print("Testing Gaussian Mixture Models...")
gmm = GaussianMixture(n_components=optimal_k, random_state=42)
gmm_labels = gmm.fit_predict(primary_embedding)
gmm_ari = adjusted_rand_score(primary_cluster_labels, gmm_labels)
method_stability_results.append({'method': 'Gaussian_Mixture', 'ARI': gmm_ari})
print(f"GMM ARI: {gmm_ari:.4f}")

# Method 4: K-means with different parameters 
print("Variants: init= k-means++")
# K-means with different initialization methods
kmeans_kmpp = KMeans(n_clusters=optimal_k, init='k-means++', random_state=42)
kmpp_labels = kmeans_kmpp.fit_predict(primary_embedding)
kmpp_ari = adjusted_rand_score(primary_cluster_labels, kmpp_labels)
method_stability_results.append({'method': 'KMeans_KMPP', 'ARI': kmpp_ari})

print("Variants: init= random")
kmeans_random = KMeans(n_clusters=optimal_k, init='random', random_state=42, n_init=10)
random_labels = kmeans_random.fit_predict(primary_embedding)
random_ari = adjusted_rand_score(primary_cluster_labels, random_labels)
method_stability_results.append({'method': 'KMeans_Random', 'ARI': random_ari})
print(f"K-means Random init ARI: {random_ari:.4f}")

# Save method stability results
method_df = pd.DataFrame(method_stability_results)
method_df.to_csv(f'{band}_RobustTest_clustering_results/method_stability_results.txt', 
                 sep='\t', index=False)
print("Saved method stability results")

What ARI (Adjusted Rand Index) Means
<br>**0.8 - 1.0** = Very strong agreement  
**0.6 - 0.8** = Strong agreement  
**0.4 - 0.6** = Moderate agreement  
**< 0.4** = Weak agreement

In [ ]:
# Step 10: Test Parameter Sensitivity
print("\n=== Step 10: Parameter Sensitivity Analysis ===")

parameter_sensitivity_results = []

# Test different k values around optimal_k
print("Testing different k values...")
for test_k in [optimal_k-1, optimal_k, optimal_k+1]:
    if test_k >= 2:  # Ensure at least 2 clusters
        kmeans_test = KMeans(n_clusters=test_k, random_state=42, n_init=10)
        test_labels = kmeans_test.fit_predict(primary_embedding)
        
        # Calculate ARI (need to handle different k - use maximum ARI over all possible label mappings)
        from scipy.optimize import linear_sum_assignment
        from sklearn.metrics import confusion_matrix
        
        # For different k values, we need a different approach
        if test_k == optimal_k:
            test_ari = adjusted_rand_score(primary_cluster_labels, test_labels)
        else:
            # For different k, we can't directly use ARI, so we'll skip or use different metric
            test_ari = np.nan
            
        parameter_sensitivity_results.append({
            'parameter': 'n_clusters', 
            'value': test_k,
            'ARI': test_ari,
            'WCSS': kmeans_test.inertia_
        })
        print(f"k={test_k}: ARI={test_ari:.4f}, WCSS={kmeans_test.inertia_:.2f}")

WCSS drops a lot from **k=3 → k=4**, then the drop **slows down from k=4 → k=5**.  This “slowing” indicates the **elbow around k=4**, which matches your `optimal_k`.

In [ ]:
# Test different UMAP parameters
print("Testing different UMAP parameters...")
test_umap_params = [
    {'n_neighbors': max(2, optimal_n_neighbors-1), 'min_dist': optimal_min_dist},
    {'n_neighbors': optimal_n_neighbors, 'min_dist': optimal_min_dist},
    {'n_neighbors': optimal_n_neighbors+1, 'min_dist': optimal_min_dist},
    {'n_neighbors': optimal_n_neighbors, 'min_dist': max(0.01, optimal_min_dist-0.1)},
    {'n_neighbors': optimal_n_neighbors, 'min_dist': min(0.99, optimal_min_dist+0.1)}
]

for i, params in enumerate(test_umap_params):
    reducer_test = UMAP(n_neighbors=params['n_neighbors'],
                       min_dist=params['min_dist'],
                       random_state=42,
                       n_components=2)
    test_embedding = reducer_test.fit_transform(X)
    
    kmeans_test = KMeans(n_clusters=optimal_k, random_state=42, n_init=10)
    test_labels = kmeans_test.fit_predict(test_embedding)
    test_ari = adjusted_rand_score(primary_cluster_labels, test_labels)
    
    parameter_sensitivity_results.append({
        'parameter': 'UMAP_params',
        'value': f"n_neighbors={params['n_neighbors']}, min_dist={params['min_dist']}",
        'ARI': test_ari,
        'WCSS': kmeans_test.inertia_
    })
    print(f"UMAP {params}: ARI={test_ari:.4f}")

In [ ]:
# Save parameter sensitivity results
param_sens_df = pd.DataFrame(parameter_sensitivity_results)
param_sens_df.to_csv(f'{band}_RobustTest_clustering_results/parameter_sensitivity_results.txt', 
                     sep='\t', index=False)
print("Saved parameter sensitivity results")

In [ ]:
# Step 10: Comprehensive Stability Summary
print("\n=== Step 10: Comprehensive Stability Summary ===")

# Calculate overall stability metrics
method_ari_mean = method_df['ARI'].mean()
method_ari_std = method_df['ARI'].std()

# Filter parameter results for same k comparisons
same_k_results = param_sens_df[param_sens_df['parameter'] == 'n_clusters']
same_k_results = same_k_results[same_k_results['value'] == optimal_k]

print(f"\n--- STABILITY SUMMARY ---")
print(f"UMAP Stability (Procrustes): {np.mean(procrustes_errors):.4f}")
print(f"Pipeline Stability (ARI): {np.mean(ari_scores):.4f} ± {np.std(ari_scores):.4f}")
print(f"Method Stability (ARI): {method_ari_mean:.4f} ± {method_ari_std:.4f}")
print(f"Consensus Agreement: {consensus_ari:.4f}")

# Interpretation guidance
print(f"\n--- INTERPRETATION ---")
if method_ari_mean > 0.7:
    print("✓ EXCELLENT: Clusters are highly robust to method choice")
elif method_ari_mean > 0.5:
    print("✓ GOOD: Clusters show reasonable robustness across methods")  
elif method_ari_mean > 0.3:
    print("○ MODERATE: Some method dependency observed")
else:
    print("○ CAUTION: High variability across clustering methods")

In [ ]:
# Save comprehensive summary
comprehensive_summary = {
    'data_shape': str(X.shape),
    'optimal_n_neighbors': optimal_n_neighbors,
    'optimal_min_dist': optimal_min_dist,
    'optimal_k': optimal_k,
    'umap_stability_procrustes': np.mean(procrustes_errors),
    'pipeline_stability_ari_mean': np.mean(ari_scores),
    'pipeline_stability_ari_std': np.std(ari_scores),
    'method_stability_ari_mean': method_ari_mean,
    'method_stability_ari_std': method_ari_std,
    'consensus_agreement_ari': consensus_ari,
    'overall_robustness': 'Excellent' if method_ari_mean > 0.7 else 
                         'Good' if method_ari_mean > 0.5 else 
                         'Moderate' if method_ari_mean > 0.3 else 'Low'
}

pd.DataFrame([comprehensive_summary]).to_csv(f'{band}_RobustTest_clustering_results/comprehensive_stability_summary.txt', 
                                           sep='\t', index=False)

print("\n=== CLUSTERING STABILITY ANALYSIS COMPLETE ===")

# ------------------------------------------ END -------------------------------------------